# Zadanie 2B — PAMAP2

**Cel:** przygotuj wielokanałowe sygnały z czujników ubieralnych, zbuduj model rozpoznający aktywność i sprawdź, jak rozwiązanie generalizuje się na osoby niewidziane podczas treningu.

In [2]:
from pathlib import Path
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from huggingface_hub import hf_hub_download
from scipy import signal
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
from torch import nn

In [2]:
DATA_DIR = Path("data/zadanie-2b")
DATA_DIR.mkdir(parents=True, exist_ok=True)

archive_path = hf_hub_download(
    repo_id="AI-MED-AGH/Recruitment-Task-2B",
    filename="original/pamap2+physical+activity+monitoring.zip",
    repo_type="dataset",
    local_dir=DATA_DIR,
)
with ZipFile(archive_path) as archive:
    archive.extractall(DATA_DIR / "pamap2")

print(f"Dane zapisano w: {DATA_DIR / 'pamap2'}")

original/pamap2+physical+activity+monito(…): reconstructing file:   0%|          |  0.00B /  688MB            

original/pamap2+physical+activity+monito(…): downloading bytes:           |  0.00B            

Dane zapisano w: data/zadanie-2b/pamap2


## A — Poznanie i przygotowanie danych

Rozpoznaj uczestników, aktywności, czujniki, częstotliwości próbkowania i braki danych. Uzasadnij wybór sygnałów, obsługę braków oraz sposób tworzenia okien. Podziel dane według uczestników i dopilnuj, aby żadna osoba nie występowała w więcej niż jednym podziale.

In [18]:
dataset_path = Path.cwd() / "data" / "zadanie-2b" / "pamap2"
dataset_zip = dataset_path / "PAMAP2_Dataset.zip"
with ZipFile(dataset_zip, "r") as f:
    f.extractall(dataset_path)

In [49]:
dataset_path = Path.cwd() / "data" / "zadanie-2b" / "pamap2" / "PAMAP2_Dataset" / "Protocol"
all_subjects = list(dataset_path.iterdir())
train_subjects = all_subjects[:6]
test_subjects = all_subjects[6:8]
val_subjects = all_subjects[8:9]
for file in all_subjects[:1]:
    df_test = pd.read_csv(file, delimiter=" ")
df_test.iloc[:,1].value_counts()

0
0     126459
1      27187
6      23575
17     23573
2      23480
16     22941
4      22253
3      21717
5      21265
7      20265
12     15890
13     14899
24     12912
Name: count, dtype: int64

In [ ]:
#informacja o czasie jest bezwartościowa w kontekście klasyfikacji aktywności, więc zostanie odrzucona.
#W readme napisano zostało również że drugi akcelerometr jest źle skalibrowany więc również zostanie odrzucony
#z każdego IMU sensora
activieties = list(range(1, 14)) + list(range(16, 21)) + [24]
print(activieties)
def preprocess_subject(file: Path):
    df = pd.read_csv(file, header=None, sep=r"\s+")
    df.interpolate("linear").ffill().bfill().fillna(0)
    df = df[df[:,1].isin(activieties)].copy()
    y = df.iloc[:, 1].values
    correct_indicies = [2]
    for i in range [3, 20, 37]:
        correct_indicies.append(list(range(i, i + 4)) + list(range(i + 4, i + 17)))
    X = df.iloc[:, correct_indicies].values
    return X, y



[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 16, 17, 18, 19, 20, 24]


## B — Modelowanie i walidacja

Wytrenuj model rozpoznawania aktywności. Przeanalizuj cechy lub architekturę, procedurę treningową i walidację z rozłącznymi uczestnikami. Wszystkie transformacje dopasowuj wyłącznie na danych treningowych, a niewidzianych uczestników testowych zachowaj do końcowej oceny.

## C — Ocena i wyjaśnialność

Oceń model na niewidzianych uczestnikach, używając odpowiednich metryk wieloklasowych oraz analizy wyników dla klas i osób. Znajdź i zacytuj metodę wyjaśniania modeli sygnałowych lub szeregów czasowych, zastosuj ją do poprawnych i błędnych predykcji kilku aktywności i omów ograniczenia.

## Oddanie zadania

Zapisz kod, wyniki, wykresy i uzasadnienia w tym notatniku. Podaj użyte źródła oraz narzędzia AI i zadbaj o odtwarzalność rozwiązania. Umieść rozwiązanie na swoim repozytorium.